# A/A calibration, A/B power, and the leaderboard

One `SimulationSpec` describes one panel and every method reads the same draw of it. A/A
calibration forces `effect = 0` and counts the runs whose interval at mass `1 − α` excludes
zero; the count is judged against the exact binomial region `clopper_pearson(n, α, 1e-3)`,
and an `Unsupported` return is counted separately (more than 10 % of them fails the method
too). A/B power scores rejections, bias, RMSE and coverage at a non-zero effect.

Under this DGP the DiD estimator's standard error is exact:
`se = noise_sd · sqrt(1/n_pre + 1/n_post) · sqrt(1/n_treated + 1/n_control)` — the unit
intercepts cancel in each unit's change and the common shocks cancel in the contrast.

In [ ]:
import numpy as np

from axiom.design import (
    CalibrationResult, Leaderboard, LeaderboardRow, METHODS, PanelDesign, SimulatedPower,
    SimulationSpec, calibrate_method, calibrate_registry, design_for_method,
    difference_in_differences_se, leaderboard, power_from_se, simulate_panel, simulated_power,
)

In [ ]:
spec = SimulationSpec(n_units=16, n_periods=12, n_pre=6, n_treated=8, noise_sd=1.0, n_simulations=100, seed=11)
print("n_post:", spec.n_post, "n_control:", spec.n_control, "alpha:", spec.alpha)
for m in METHODS:
    d: PanelDesign = design_for_method(m)
    print(f"{m:28s} -> {d}")
panel = simulate_panel(spec, np.random.default_rng(0), design="holdout")
print("one panel:", panel.outcome.shape)

In [ ]:
cal: CalibrationResult = calibrate_method("difference_in_differences", spec, alpha=0.05)
print(f"false positives {cal.false_positive_count}/{cal.n_evaluated} = {cal.false_positive_rate:.3f}")
print(f"acceptance region [{cal.region.lower}, {cal.region.upper}] at nominal {cal.alpha}; passed={cal.passed} {cal.reason}")

In [ ]:
se = difference_in_differences_se(spec)
predicted = power_from_se(0.8, se).power
ab = spec.model_copy(update={"effect": 0.8})
sp: SimulatedPower = simulated_power("difference_in_differences", ab, predicted_power=predicted)
print(f"exact DiD se {se:.4f}; predicted power {predicted:.3f}; realized {sp.power:.3f} ({sp.rejections}/{sp.n_evaluated})")
print(f"bias {sp.bias:+.3f}  rmse {sp.rmse:.3f}  coverage {sp.coverage:.2f}  within prediction: {sp.within_prediction}")

## Calibrating the registry

`calibrate_registry` returns a *new* read-only registry with each method's status set from its
A/A result; `METHODS` is never mutated. `leaderboard` ranks methods by mean power across effect
sizes at their calibrated size, calibrated methods first.

In [ ]:
small = spec.model_copy(update={"n_simulations": 40})
registry, results = calibrate_registry(small, alpha=0.05)
for r in results:
    print(f"{r.method:28s} design={r.design:10s} fp={r.false_positive_count}/{r.n_evaluated} passed={r.passed} -> {registry[r.method].status}")

In [ ]:
board: Leaderboard = leaderboard(small, effects=(0.8, 1.6), alpha=0.05)
for row in board.rows:
    assert isinstance(row, LeaderboardRow)
    print(f"#{row.rank} {row.method:28s} calibrated={row.calibrated!s:5} powers={tuple(round(p, 2) for p in row.powers)} mean={row.mean_power:.2f}")
print(board.row("difference_in_differences").coverages)